In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [10]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="ollama:llama3.1:8b",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [12]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="You MUST use your tools. First, use read_email to read my email. Then, immediately use send_email to send a reply agreeing to reschedule.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [13]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'I '
                                                                          'have '
                                                                          'reviewed '
                                                                          'your '
                                                                          'email '
                                                                          'and '
                                                                          'agree '
                                                                          'to '
                                                                          'reschedule. '
                                                                          'Please '
                                                                          'let '
                                                                          'me '
          

In [14]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'I have reviewed your email and agree to reschedule. Please let me know a suitable time and I will make sure to adjust my schedule accordingly.'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'I have reviewed your email and agree to reschedule. Please let me know a suitable time and I will make sure to adjust my schedule accordingly.'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='79bef6d6910c3a9fac392f16539faf15')]


In [15]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

I have reviewed your email and agree to reschedule. Please let me know a suitable time and I will make sure to adjust my schedule accordingly.


## Approve

In [18]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='80f7a824-318d-43c4-a438-8554341c7a4a'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:8b', 'created_at': '2026-09-15T20:36:19.7770259Z', 'done': True, 'done_reason': 'stop', 'total_duration': 15340910800, 'load_duration': 10437258200, 'prompt_eval_count': 210, 'prompt_eval_duration': 521663000, 'eval_count': 34, 'eval_duration': 4292868000, 'logprobs': None, 'model_name': 'llama3.1:8b', 'model_provider': 'ollama'}, id='lc_run--01a0a6c8-accf-7322-b271-3ab7dbfbf872-0', tool_calls=[{'name': 'send_email', 'args': {'body': 'Please find the response to your email below. I will send it immediately in this thread.'}, 'id': '958dd247-975a-484c-b668-bf79aa288f

## Reject

In [17]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'I '
                                                                          'have '
                                                                          'reviewed '
                                                                          'your '
                                                                          'email '
                                                                          'and '
                                                                          'agree '
                                                                          'to '
                                                                          'reschedule. '
                                                                          'Please '
                                                                          'let '
                                                                          'me '
          

In [ ]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

## Edit

In [ ]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)